
---
Methods Project Part 3
---


Using previously used functions from part 2 and imports

In [7]:
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score
import numpy as np
import plotly.graph_objects as go

from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from keras.datasets import mnist
from keras import layers, models
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping

RANDOM_STATE = 1 

(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = mnist.load_data()

# We combine the train and test set to a single before masking
X_mnist = np.concatenate([X_train_mnist, X_test_mnist], axis=0)
y_mnist = np.concatenate([y_train_mnist, y_test_mnist], axis=0)

# We reshape the data to be in the format (num_samples, num_features) and normalize pixel values to [0, 1]
image_size = X_train_mnist.shape[1] #28 for MNIST
# (num_samples, 28, 28) -> (num_samples, 28, 28, 1) and normalize pixel values to [0, 1]
X_mnist = X_mnist.reshape(-1, image_size, image_size, 1).astype('float32') / 255

# Dataset A (digits 0-4), Dataset B (digits 5-9)
X_mnist_A,y_mnist_A = X_mnist[y_mnist <= 4], y_mnist[y_mnist <= 4]

# One-hot encoding for the labels
y_mnist_A = to_categorical(y_mnist_A, num_classes=5)

X_mnist_A_train, X_mnist_A_test, y_mnist_A_train, y_mnist_A_test = train_test_split(X_mnist_A, y_mnist_A, test_size=0.2, random_state=RANDOM_STATE)

def get_2_layer_cnn_base():
    base = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten()
    ], name="base_2_layer_cnn")
    return base

def build_and_compile_full_model(base_model):
    """Attaches 2 Fully-Connected layers"""
    inputs = layers.Input(shape=(28, 28, 1))
    # Pass the inputs through the base feature extractor
    x = base_model(inputs)
    
    # Classification head with 2 fully connected layers
    x = layers.Dense(64, activation='relu', name="fc_1")(x)
    outputs = layers.Dense(5, activation='softmax', name="fc_2_out")(x)
    
    # Combine into a single model
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

early_stopping = EarlyStopping(
    monitor='val_loss',         
    patience=3,                 
    restore_best_weights=True,  # Revert to the best model, not the last epoch's model
    verbose=1              
)

In [8]:
print("Training 2-layer CNN on 80% of randomly sampled Dataset A")
base_2 = get_2_layer_cnn_base()
model_2_data_A = build_and_compile_full_model(base_2)
model_2_data_A.fit(X_mnist_A, y_mnist_A, epochs=100, batch_size=128, verbose="auto", validation_split=0.1,callbacks=[early_stopping])


Training 2-layer CNN on 80% of randomly sampled Dataset A
Epoch 1/100
252/252 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9639 - loss: 0.1341 - val_accuracy: 0.9919 - val_loss: 0.0295
Epoch 2/100
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9915 - loss: 0.0281 - val_accuracy: 0.9978 - val_loss: 0.0083
Epoch 3/100
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9939 - loss: 0.0197 - val_accuracy: 0.9944 - val_loss: 0.0151
Epoch 4/100
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9964 - loss: 0.0127 - val_accuracy: 0.9980 - val_loss: 0.0057
Epoch 5/100
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9972 - loss: 0.0097 - val_accuracy: 0.9989 - val_loss: 0.0030
Epoch 6/100
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9978 - loss: 0.0080 - val_accuracy: 0.9994 - val_loss: 0.0027
Epoch 7/100
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9985 - loss: 0.0051 - val_accuracy: 0.9994 - val_loss: 0.0027
Epoch 8/100
252/252 ━━━━━━━━━━━━━━━━━━━